#  Detectron2 Mask R-CNN Trainer

This notebook provides a complete and professional pipeline for fine-tuning the **Detectron2 Mask R-CNN (R50-FPN 3x)** model on the custom microscopic rock cutting dataset.

###  Model Overview
The model `maskrcnnR50FPN3x` is obtained from the **Detectron2 Model Zoo**.
- **100% Pretrained**: The ResNet-50 backbone is pretrained on ImageNet, and the entire Mask R-CNN architecture is pretrained on the COCO instance segmentation dataset (80 categories).
- **Fine-tuning**: We fine-tune this robust model on our specific 6-7 rock classes, exploiting generalized visual features which significantly accelerates convergence and improves performance despite the smaller dataset size.

###  Prerequisites
Detectron2 specifically requires **COCO JSON format**.
If your dataset is currently in YOLO format, you must first convert it using the `convertyolotococo.py` script located in `scripts/datapreprocessing/singlemodel/`.

---

## 1. Environment Setup
Install the required PyTorch and Detectron2 libraries.

In [ ]:
!pip install ninja
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation
!pip install opencv-python matplotlib

In [ ]:
import torch
import detectron2
from detectron2.utils.logger import setup_logger

setup_logger()
print(f"PyTorch Version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")
print(f"Detectron2 Version: {detectron2.__version__}")

## 2. Dataset Registration
Register the mapped COCO datasets for `train` and `val` configurations.

In [ ]:
import os
from detectron2.data.datasets import register_coco_instances

# Ensure you have converted YOLO formats properly and updated paths here.
# Example paths, replace with actual converted absolute dataset root paths.
DATASET_ROOT = "../../datasets/batch4_coco"  

train_json = os.path.join(DATASET_ROOT, "train/annotations/instances_default.json")
train_imgs = os.path.join(DATASET_ROOT, "train/images")

val_json = os.path.join(DATASET_ROOT, "val/annotations/instances_default.json")
val_imgs = os.path.join(DATASET_ROOT, "val/images")

try:
    register_coco_instances("rock_train", {}, train_json, train_imgs)
    register_coco_instances("rock_val", {}, val_json, val_imgs)
    print("Datasets registered successfully.")
except Exception as e:
    print(f"Warning or Registration Error: {e}")

## 3. Training Configuration
Define the custom trainer to initialize the parameters for Mask R-CNN, including batch sizes linearly scaling across memory availability.

In [ ]:
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()

# Load baseline Mask R-CNN config
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))

# Pretrained model checkpoint URL
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

# Assign dataset configurations
cfg.DATASETS.TRAIN = ("rock_train",)
cfg.DATASETS.TEST = ("rock_val",)

# Multiprocessing settings
cfg.DATALOADER.NUM_WORKERS = 4

# Batch configurations (Modify IMS_PER_BATCH considering your GPU VRAM)
cfg.SOLVER.IMS_PER_BATCH = 4       # Increase to 8 or 16 if >= 24GB VRAM
cfg.SOLVER.BASE_LR = 0.0001        # Specifically lowered for class-imbalanced rock instances
cfg.SOLVER.MAX_ITER = 3000         # Total iterations, usually 1000-5000 is adequate
cfg.SOLVER.STEPS = []              # Custom decay steps if needed (e.g., (2000, 2500))

# Architecture configs
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512  # RoI bounding box batch processing constraints
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6             # Update to 6 specific rock classes
cfg.MODEL.ROI_MASK_HEAD.BOUNDARY_PRECISION = True # Better delineations for irregular structures

cfg.INPUT.RANDOM_FLIP = "horizontal"          # Built-in robust augmentation mechanisms

# Save configurations
import string, random
random_slug = "".join(random.choices(string.ascii_letters + string.digits, k=6))
RUNNER_NAME = f"Detectron2_Batch4_{random_slug}"
cfg.OUTPUT_DIR = f"./models/{RUNNER_NAME}"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

## 4. Run Training Pipeline
Initialize the `DefaultTrainer`, load the previously processed configs, and map the outputs appropriately.

In [ ]:
from detectron2.engine import DefaultTrainer
import logging
from detectron2.evaluation import inference_on_dataset
from detectron2.data import build_detection_test_loader
import os

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        from detectron2.evaluation import COCOEvaluator
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        os.makedirs(output_folder, exist_ok=True)
        # Natively compute overall and per-class AP/AR metrics against validation splits
        # to closely monitor minority class performance.
        return COCOEvaluator(dataset_name, output_dir=output_folder)

print(f"\n{'='*50}")
print(f" INITIATING MASK R-CNN TRAINING RUN")
print(f"{'='*50}\n")

trainer = CustomTrainer(cfg)
# False means it initializes from baseline pretrained weights
trainer.resume_or_load(resume=False)

# Execute the main training loop
# Note: Standard output suppresses verbose metrics. Review TensorBoard or model output directory for progressive history metrics.
try:
    trainer.train()

    print(f"\n{'='*50}")
    print(f"  PER-CLASS EVALUATION (MINORITY CLASS MONITORING)")
    print(f"{'='*50}")
    evaluator = CustomTrainer.build_evaluator(cfg, "rock_val")
    val_loader = build_detection_test_loader(cfg, "rock_val")
    eval_results = inference_on_dataset(trainer.model, val_loader, evaluator)
    print(eval_results)
    print(f"{'='*50}\n")
    print(f"\n Training sequence completed perfectly. Best checkpoints preserved at {cfg.OUTPUT_DIR}.")
except Exception as e:
    print(f" Training failed with exception: {e}")

## 5. Visual Testing / Fast Inference (Optional)

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
import cv2
import matplotlib.pyplot as plt
import random

def test_inference(test_image_dir):
    cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5   # Target 50% inference threshold strictly for visual rendering
    predictor = DefaultPredictor(cfg)

    images = [img for img in os.listdir(test_image_dir) if img.endswith((".jpg", ".png"))]
    if not images:
        print("No images for testing available.")
        return
        
    sample_img = os.path.join(test_image_dir, random.choice(images))
    im = cv2.imread(sample_img)
    
    print(f"Running predictor algorithms on: {sample_img}")
    outputs = predictor(im)
    
    v = Visualizer(im[:, :, ::-1], MetadataCatalog.get("rock_val"), scale=1.2)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    
    plt.figure(figsize=(14, 10))
    plt.imshow(out.get_image())
    plt.axis("off")
    plt.title("Mask R-CNN Prediction")
    plt.show()

# Ensure execution is called properly when checking local runs
test_inference(val_imgs)